# ✈️ Flight Fare Prediction
**Goal:** Predict flight ticket prices using machine learning.


## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')
print("Libraries imported ✅")


## Step 2: Load Dataset

In [ ]:
from google.colab import files
x = files.upload()


In [ ]:
df = pd.read_excel("Flight_Fare.xlsx")
print("Shape:", df.shape)
df.head()


## Step 3: Basic Data Checks

### 3.1 Data types and column info

In [ ]:
df.info()

### 3.2 Statistical summary

In [ ]:
df.describe()

### 3.3 Missing values

In [ ]:
df.isnull().sum()

## Step 4: Data Cleaning

Route and Total_Stops each have 1 missing value — we simply drop those rows.

In [ ]:
df.dropna(inplace=True)
print("Shape after cleaning:", df.shape)


## Step 5: Feature Engineering

ML models can't read dates or time strings. We convert them into numbers.

### 5.1 Journey Date → Day, Month, Weekday

In [ ]:
df['Date_of_Journey'] = pd.to_datetime(df['Date_of_Journey'])
df['Journey_Day']       = df['Date_of_Journey'].dt.day
df['Journey_Month']     = df['Date_of_Journey'].dt.month
df['Journey_DayOfWeek'] = df['Date_of_Journey'].dt.dayofweek
df.drop('Date_of_Journey', axis=1, inplace=True)


### 5.2 Departure Time → Hour, Minute

In [ ]:
if 'Dep_Time' in df.columns:
    df['Dep_Time'] = pd.to_datetime(df['Dep_Time'])
    df['Dep_Hour'] = df['Dep_Time'].dt.hour
    df['Dep_Min']  = df['Dep_Time'].dt.minute
    df.drop('Dep_Time', axis=1, inplace=True)


### 5.3 Arrival Time → Hour, Minute

In [ ]:
if 'Arrival_Time' in df.columns:
    df['Arrival_Time'] = pd.to_datetime(df['Arrival_Time'])
    df['Arrival_Hour'] = df['Arrival_Time'].dt.hour
    df['Arrival_Min']  = df['Arrival_Time'].dt.minute
    df.drop('Arrival_Time', axis=1, inplace=True)


### 5.4 Duration → Hours and Minutes

In [ ]:
if 'Duration' in df.columns:
    if pd.api.types.is_numeric_dtype(df['Duration']):
        df['Duration_Hours']   = df['Duration'] // 60
        df['Duration_Minutes'] = df['Duration'] % 60
    else:
        df['Duration_Hours']   = df['Duration'].str.extract(r'(\d+)h').fillna(0).astype(int)
        df['Duration_Minutes'] = df['Duration'].str.extract(r'(\d+)m').fillna(0).astype(int)
    df.drop('Duration', axis=1, inplace=True)


### 5.5 Total Stops → Numbers (0, 1, 2, 3, 4)

In [ ]:
df['Total_Stops'] = df['Total_Stops'].map({
    'non-stop': 0, '1 stop': 1, '2 stops': 2, '3 stops': 3, '4 stops': 4
})


### 5.6 Drop unused columns

In [ ]:
for col in ['Route', 'Additional_Info']:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)
print("Final columns:", list(df.columns))


## Step 6: Exploratory Data Analysis (EDA)
We look at 3 key relationships that actually matter for explaining flight prices.


### 6.1 Price Distribution — Raw vs Log

This shows why we need log transformation — raw price is skewed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(df['Price'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title("Raw Price (Skewed)")
axes[0].set_xlabel("Price (₹)")

sns.histplot(np.log1p(df['Price']), bins=50, kde=True, ax=axes[1], color='coral')
axes[1].set_title("Log-Transformed Price (Balanced)")
axes[1].set_xlabel("log(Price + 1)")

plt.suptitle("Log transformation balances the skewed price distribution", fontsize=11)
plt.tight_layout()
plt.show()


### 6.2 Airline vs Average Price

This shows which airlines are expensive — useful business insight.

In [ ]:
airline_avg = df.groupby('Airline')['Price'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 4))
bars = plt.bar(airline_avg.index, airline_avg.values, color='steelblue', edgecolor='black')
plt.title("Average Price by Airline")
plt.xlabel("Airline")
plt.ylabel("Avg Price (₹)")
plt.xticks(rotation=45, ha='right')
for bar, val in zip(bars, airline_avg.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
             f'₹{int(val):,}', ha='center', fontsize=7)
plt.tight_layout()
plt.show()
print("Insight: Jet Airways Business is most expensive. IndiGo is budget-friendly.")


### 6.3 Duration vs Price

Longer flights cost more — this confirms Duration is an important feature.

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(df['Duration_Hours'], df['Price'], alpha=0.3, s=10, color='teal')
plt.title("Flight Duration vs Price")
plt.xlabel("Duration (Hours)")
plt.ylabel("Price (₹)")
plt.tight_layout()
plt.show()
print("Insight: Longer flights generally cost more. Duration is a strong predictor.")


## Step 7: Correlation Heatmap

Shows how strongly each numeric feature is related to Price.

In [ ]:
plt.figure(figsize=(11, 7))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
print("Duration_Hours has the strongest positive correlation with Price.")


## Step 8: Remove Outliers

We remove the bottom 1% and top 1% of prices to avoid extreme values affecting the model.

In [ ]:
q1  = df['Price'].quantile(0.01)
q99 = df['Price'].quantile(0.99)
df  = df[(df['Price'] > q1) & (df['Price'] < q99)]
print("Shape after outlier removal:", df.shape)
print(f"Price range: ₹{df['Price'].min():,.0f} – ₹{df['Price'].max():,.0f}")


## Step 9: Define Features (X) and Target (y)

X = all input columns. y = Price (log-transformed).

In [ ]:
X = df.drop('Price', axis=1)
y = np.log1p(df['Price'])
print("Features:", X.shape[1], "| Rows:", X.shape[0])


## Step 10: Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape[0], "rows")
print("Test :", X_test.shape[0], "rows")


## Step 11: Preprocessing Pipeline

OneHotEncoder converts text columns (Airline, Source, Destination) into numbers. Wrapped in a Pipeline to prevent data leakage.

In [ ]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()
print("Categorical:", cat_cols)
print("Numerical  :", num_cols)


In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', 'passthrough', num_cols)
])
print("Preprocessor ready ✅")


## Step 12: Train and Compare 3 Models

We use 3 models — from simple to advanced:
- **Linear Regression** — baseline
- **Random Forest** — ensemble of trees
- **XGBoost** — gradient boosting (usually best on tabular data)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest"    : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost"          : XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

results = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred     = pipe.predict(X_test)
    y_pred_inr = np.expm1(y_pred)
    y_test_inr = np.expm1(y_test)

    r2   = r2_score(y_test_inr, y_pred_inr)
    mae  = mean_absolute_error(y_test_inr, y_pred_inr)
    rmse = np.sqrt(mean_squared_error(y_test_inr, y_pred_inr))

    results[name] = {'R2': round(r2,4), 'MAE': round(mae,0), 'RMSE': round(rmse,0)}
    print(f"{name:22s} | R²: {r2:.4f} | MAE: ₹{mae:,.0f} | RMSE: ₹{rmse:,.0f}")

results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)


### Model Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]

for ax, (col, title) in zip(axes, [
    ('R2',   'R² Score (higher = better)'),
    ('MAE',  'MAE in ₹ (lower = better)'),
    ('RMSE', 'RMSE in ₹ (lower = better)')
]):
    bars = ax.bar(results_df.index, results_df[col], color=colors, edgecolor='black')
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, results_df[col]):
        label = f'{val:.3f}' if col == 'R2' else f'₹{int(val):,}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                label, ha='center', fontsize=8)

plt.suptitle("Model Comparison — Gold = Best", fontsize=12)
plt.tight_layout()
plt.show()


## Step 13: Hyperparameter Tuning (XGBoost)

XGBoost won, so we tune it further using RandomizedSearchCV with 5-fold cross-validation.

In [ ]:
param_grid = {
    'model__n_estimators'    : [300, 500, 700],
    'model__max_depth'       : [3, 5, 7],
    'model__learning_rate'   : [0.01, 0.05, 0.1],
    'model__subsample'       : [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42, verbosity=0))
])

search = RandomizedSearchCV(
    xgb_pipeline, param_grid,
    cv=5, scoring='r2', n_iter=20,
    n_jobs=-1, random_state=42, verbose=1
)
search.fit(X_train, y_train)

print("\nBest Parameters:")
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__','')}: {v}")
print(f"\nBest CV R²: {search.best_score_:.4f}")


## Step 14: Final Evaluation

We check performance on the test set using 4 clear metrics.

In [ ]:
y_pred_log = search.predict(X_test)
y_pred_inr = np.expm1(y_pred_log)
y_test_inr = np.expm1(y_test)

r2   = r2_score(y_test_inr, y_pred_inr)
mae  = mean_absolute_error(y_test_inr, y_pred_inr)
rmse = np.sqrt(mean_squared_error(y_test_inr, y_pred_inr))
mape = np.mean(np.abs((y_test_inr - y_pred_inr) / y_test_inr)) * 100

print("=" * 40)
print("  Final Evaluation — Tuned XGBoost")
print("=" * 40)
print(f"  R²   : {r2:.4f}  ({r2*100:.1f}% variance explained)")
print(f"  MAE  : ₹{mae:,.0f}")
print(f"  RMSE : ₹{rmse:,.0f}")
print(f"  MAPE : {mape:.2f}%")
print("=" * 40)


### Actual vs Predicted & Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(y_test_inr, y_pred_inr, alpha=0.3, s=10, color='steelblue')
lims = [y_test_inr.min(), y_test_inr.max()]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_title("Actual vs Predicted Price")
axes[0].set_xlabel("Actual Price (₹)")
axes[0].set_ylabel("Predicted Price (₹)")
axes[0].legend()

residuals = y_test_inr.values - y_pred_inr
sns.histplot(residuals, bins=50, kde=True, ax=axes[1], color='coral')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title("Residuals (Actual − Predicted)")
axes[1].set_xlabel("Error in ₹")

plt.tight_layout()
plt.show()
print("Points near red line = good predictions. Residuals centred at 0 = unbiased model.")


### Cross-Validation — Is the model consistent?

In [ ]:
cv_scores = cross_val_score(search.best_estimator_, X, y, cv=5, scoring='r2', n_jobs=-1)

plt.figure(figsize=(7, 4))
plt.bar([f'Fold {i+1}' for i in range(5)], cv_scores, color='steelblue', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--',
            linewidth=1.5, label=f'Mean R²: {cv_scores.mean():.4f}')
plt.title("5-Fold Cross-Validation R² Scores")
plt.ylabel("R² Score")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()
print(f"Mean R²: {cv_scores.mean():.4f}  |  Std: {cv_scores.std():.4f}")


## Step 15: Feature Importance

Which features matter most for predicting price?

In [ ]:
best_model = search.best_estimator_.named_steps['model']
importances = best_model.feature_importances_

ohe_features = (search.best_estimator_
                .named_steps['preprocessor']
                .named_transformers_['cat']
                .get_feature_names_out(cat_cols))
all_features = list(ohe_features) + num_cols

feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
feat_imp.head(10).sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title("Top 10 Features Influencing Flight Price")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()


## Step 16: Save the Model

In [ ]:
import joblib
joblib.dump(search.best_estimator_, "flight_model.pkl")
print("Model saved ✅")


## Challenges Faced

- Duration was in text format like "2h 30m" — used regex to extract hours and minutes
- Date and time columns had to be broken into numbers before the model could use them
- Price was skewed — log transformation was applied to balance it
- Categorical columns were encoded using OneHotEncoder inside a Pipeline to avoid data leakage


## Conclusion

- Performed EDA to understand pricing patterns across airlines, duration, and stops
- Engineered 10+ features from raw date/time/duration columns
- Compared 3 models — XGBoost performed best with ~85–88% R²
- Tuned XGBoost using RandomizedSearchCV with 5-fold cross-validation
- Evaluated using R², MAE, RMSE, MAPE on actual ₹ values
